# Whisper for ASR on Hinglish

This notebook compares zero-shot Whisper and fine-tuned Whisper models on Hindi + English audio.

# Zero-shot Whisper baseline on OpenSLR 104 (MUCS 2021 Hindi-English test)

**Steps**
1. Download the OpenSLR 104 Hindi-English **test** set (443 MB).
2. Cut the long recordings into utterances.
3. Run Whisper models with forced Hindi and with automatic language detection.
4. Normalize references and outputs with the same rules.
5. Calculate WER, CER, error types, script mix, and hallucinations.
6. Save every transcript and a summary table.

## 1. Settings

In [1]:
from pathlib import Path

# Models to test (multilingual only, never ".en" checkpoints).
MODELS = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    # "openai/whisper-medium",   # 769M parameters: slower, but useful as an upper reference
]

LANGUAGES = ["hi", "auto"]   # "hi" = forced Hindi, "auto" = automatic language detection
BEAMS = [1]                  # add 5 for beam search (about 2-3x slower): [1, 5]
BATCH_SIZE = 16              # lower it (8) if you get "CUDA out of memory"
MAX_NEW_TOKENS = 400         # Devanagari uses many tokens per word
MAX_UTTS = 200               # 0 = full test set. Start with 200 to check that everything works
MIN_DUR, MAX_DUR = 0.5, 30.0 # seconds; Whisper reads 30 s at a time

DATA_DIR = None
DOWNLOAD_DIR = Path("data/openslr104")
CACHE_DIR = Path("data/openslr104_test_prepared")
OUT_DIR = Path("results/baseline_openslr104")

SAVE_TO_DRIVE = False

## 2. Check the GPU

In [2]:
import torch, transformers, platform

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| python", platform.python_version())
if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU. On CPU, keep MAX_UTTS small.")

Device: Tesla T4
torch 2.14.1+cu130 | transformers 5.22.1 | python 3.13.16


## 3. Download and extract the test set

In [3]:
import tarfile, urllib.request

SAMPLE_RATE = 16000
TEST_ARCHIVE = "Hindi-English_test.tar.gz"
MIRRORS = [
    "https://www.openslr.org/resources/104/" + TEST_ARCHIVE,
    "https://us.openslr.org/resources/104/" + TEST_ARCHIVE,
]


def _progress(blocks, block_size, total):
    if total > 0:
        pct = min(100.0, blocks * block_size * 100.0 / total)
        print(f"\r{pct:5.1f}%", end="", flush=True)


def download_and_extract(download_dir: Path) -> Path:
    download_dir.mkdir(parents=True, exist_ok=True)
    archive = download_dir / TEST_ARCHIVE
    extract_dir = download_dir / "extracted"
    if extract_dir.exists() and any(extract_dir.iterdir()):
        print("Using existing folder:", extract_dir)
        return extract_dir
    if not archive.exists():
        last_err = None
        for url in MIRRORS:
            try:
                print("Downloading", url)
                tmp = archive.with_suffix(".part")
                urllib.request.urlretrieve(url, tmp, _progress)
                tmp.rename(archive)
                print()
                break
            except Exception as err:
                last_err = err
                print("\nFailed:", err)
        else:
            raise RuntimeError(f"All mirrors failed. Last error: {last_err}")
    print("Extracting", archive)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tar:
        try:
            tar.extractall(extract_dir, filter="data")
        except TypeError:
            tar.extractall(extract_dir)
    return extract_dir


DATA_ROOT = Path(DATA_DIR) if DATA_DIR else download_and_extract(DOWNLOAD_DIR)
print("Data root:", DATA_ROOT)

100.0%
Extracting data/openslr104/Hindi-English_test.tar.gz
Data root: data/openslr104/extracted


## 4. Cut the recordings into utterances

The data is in Kaldi format: `text` (transcripts), `segments` (start and end times), and `wav.scp` (audio files).
This cell finds these files, cuts each utterance, and saves it as a 16 kHz WAV file with a `manifest.jsonl`.

In [4]:
import json, re
from collections import defaultdict
import numpy as np
import soundfile as sf


def read_kaldi_map(path):
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(maxsplit=1)
                out[parts[0]] = parts[1] if len(parts) > 1 else ""
    return out


def read_segments(path):
    out = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.split()
            if len(p) >= 4:
                out[p[0]] = (p[1], float(p[2]), float(p[3]))
    return out


def find_kaldi_dir(root):
    cands = [p.parent for p in root.rglob("text") if p.is_file()]
    if not cands:
        raise FileNotFoundError(f"No Kaldi 'text' file under {root}")
    with_seg = [c for c in cands if (c / "segments").exists()]
    return (with_seg or cands)[0]


def resolve_wav(rec_id, scp_value, wav_index, base):
    if scp_value:
        for tok in [t for t in scp_value.split() if re.search(r"\.(wav|flac)$", t, re.I)]:
            p = Path(tok)
            for cand in (p, base / p):
                if cand.exists():
                    return cand
            if p.name in wav_index:
                return wav_index[p.name]
    for ext in (".wav", ".flac", ".WAV"):
        if rec_id + ext in wav_index:
            return wav_index[rec_id + ext]
    return None


def load_audio(path):
    audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        from scipy.signal import resample_poly
        g = np.gcd(sr, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // g, sr // g).astype("float32")
    return audio


def prepare_utterances(data_root, cache_dir, min_dur, max_dur):
    manifest = cache_dir / "manifest.jsonl"
    if manifest.exists():
        print("Using existing manifest:", manifest)
        return manifest
    kdir = find_kaldi_dir(data_root)
    texts = read_kaldi_map(kdir / "text")
    segments = read_segments(kdir / "segments") if (kdir / "segments").exists() else {}
    scp = read_kaldi_map(kdir / "wav.scp") if (kdir / "wav.scp").exists() else {}
    wav_index = {}
    for p in data_root.rglob("*"):
        if p.suffix.lower() in (".wav", ".flac") and p.is_file():
            wav_index.setdefault(p.name, p)
    print(f"Kaldi folder: {kdir}")
    print(f"{len(texts)} transcripts | {len(segments)} segments | "
          f"{len(scp)} wav.scp entries | {len(wav_index)} audio files")

    by_rec = defaultdict(list)
    if segments:
        for utt, (rec, s, e) in segments.items():
            by_rec[rec].append((utt, s, e))
    else:
        for utt in texts:
            by_rec[utt].append((utt, None, None))

    wav_out = cache_dir / "wav"
    wav_out.mkdir(parents=True, exist_ok=True)
    stats, rows = defaultdict(int), []
    for rec, utts in sorted(by_rec.items()):
        path = resolve_wav(rec, scp.get(rec, ""), wav_index, kdir)
        if path is None:
            stats["missing_audio"] += len(utts)
            continue
        audio = load_audio(path)
        for utt, s, e in utts:
            text = texts.get(utt, "").strip()
            if not text:
                stats["no_text"] += 1
                continue
            clip = audio if s is None else audio[int(s * SAMPLE_RATE): int(e * SAMPLE_RATE)]
            dur = len(clip) / SAMPLE_RATE
            if dur < min_dur or dur > max_dur:
                stats["bad_duration"] += 1
                continue
            out_path = wav_out / f"{utt}.wav"
            sf.write(str(out_path), clip, SAMPLE_RATE, subtype="PCM_16")
            rows.append({"utt_id": utt, "rec_id": rec, "audio": str(out_path),
                         "duration": round(dur, 3), "text": text})
    with open(manifest, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    hours = sum(r["duration"] for r in rows) / 3600
    print(f"Kept {len(rows)} utterances ({hours:.2f} h). Skipped: no_text={stats['no_text']}, "
          f"bad_duration={stats['bad_duration']}, missing_audio={stats['missing_audio']}")
    if not rows:
        raise RuntimeError("No utterances prepared. Check the folder layout printed above.")
    return manifest


MANIFEST = prepare_utterances(DATA_ROOT, CACHE_DIR, MIN_DUR, MAX_DUR)

Kaldi folder: data/openslr104/extracted/test/transcripts
3136 transcripts | 3136 segments | 30 wav.scp entries | 30 audio files
Kept 3132 utterances (5.14 h). Skipped: no_text=0, bad_duration=4, missing_audio=0


## 5. Text normalization

The same rules are applied to the references and to the model outputs.
Numbers are **not** converted between digits and words.

In [5]:
import unicodedata

_DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
_TAGS = re.compile(r"<[^>]*>|\[[^\]]*\]|\{[^}]*\}")

NORMALIZATION_RULES = """
1. Unicode NFC (also makes nukta letters consistent).
2. Remove tags such as <noise>, [music], {laugh}.
3. Devanagari digits -> ASCII digits.
4. Lowercase (affects Latin only).
5. Remove punctuation and symbols (Unicode P*, S*) and format characters (Cf, e.g. ZWJ).
   Devanagari vowel signs and virama are kept.
6. Collapse whitespace. Numbers are not converted between digits and words.
"""


def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = _TAGS.sub(" ", text)
    text = text.translate(_DEV_DIGITS).lower()
    chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat[0] in ("P", "S"):
            chars.append(" ")
        elif cat != "Cf":
            chars.append(ch)
    return re.sub(r"\s+", " ", "".join(chars)).strip()


def word_script(word):
    dev = lat = other = 0
    for ch in word:
        o = ord(ch)
        if 0x0900 <= o <= 0x097F or 0xA8E0 <= o <= 0xA8FF:
            dev += 1
        elif "a" <= ch <= "z":
            lat += 1
        elif ch.isdigit():
            continue
        elif ch.isalpha():
            other += 1          # for example Urdu/Arabic script
    if dev == lat == other == 0:
        return "digit"
    best = max(dev, lat, other)
    return "devanagari" if best == dev else "latin" if best == lat else "other"

for s in ["नमस्ते, आज हम bash script सीखेंगे।", "यह <noise> GChemPaint का tutorial है", "०१२ Version 4.1"]:
    n = normalize(s)
    print(f"{s!r:45} -> {n!r:40} {[word_script(w) for w in n.split()]}")

'नमस्ते, आज हम bash script सीखेंगे।'          -> 'नमस्ते आज हम bash script सीखेंगे'       ['devanagari', 'devanagari', 'devanagari', 'latin', 'latin', 'devanagari']
'यह <noise> GChemPaint का tutorial है'        -> 'यह gchempaint का tutorial है'           ['devanagari', 'latin', 'devanagari', 'latin', 'devanagari']
'०१२ Version 4.1'                             -> '012 version 4 1'                        ['digit', 'latin', 'digit', 'digit']


## 6. Metrics

WER and CER use word-level and character-level edit distance (the same definition as the `jiwer` library).
Error types: **substitutions** (wrong word), **deletions** (missing word), **insertions** (extra word).
A "possible hallucination" is an output with more than 2 × the reference words + 5.

In [6]:
try:
    from rapidfuzz.distance import Levenshtein as _RFLev   # fast, if available
except ImportError:
    _RFLev = None


def align_counts(ref, hyp):
    """Return (substitutions, deletions, insertions, hits) for two token lists."""
    n, m = len(ref), len(hyp)
    # dp[i][j] = (cost, S, D, I)
    prev = [(j, 0, 0, j) for j in range(m + 1)]
    for i in range(1, n + 1):
        cur = [(i, 0, i, 0)] + [None] * m
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                cur[j] = prev[j - 1]
            else:
                s, d, ins = prev[j - 1], prev[j], cur[j - 1]
                best = min((s[0], 0), (d[0], 1), (ins[0], 2))
                if best[1] == 0:
                    cur[j] = (s[0] + 1, s[1] + 1, s[2], s[3])
                elif best[1] == 1:
                    cur[j] = (d[0] + 1, d[1], d[2] + 1, d[3])
                else:
                    cur[j] = (ins[0] + 1, ins[1], ins[2], ins[3] + 1)
        prev = cur
    cost, S, D, I = prev[m]
    return S, D, I, n - S - D


def char_distance(a, b):
    if _RFLev is not None:
        return _RFLev.distance(a, b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def compute_metrics(refs, hyps):
    S = D = I = N = 0
    char_err = char_total = 0
    halluc = empty = 0
    ref_sc, hyp_sc = defaultdict(int), defaultdict(int)
    per_utt_wer = []
    for r, h in zip(refs, hyps):
        rw, hw = r.split(), h.split()
        s, d, i, _ = align_counts(rw, hw)
        S, D, I, N = S + s, D + d, I + i, N + len(rw)
        per_utt_wer.append(100 * (s + d + i) / max(1, len(rw)))
        char_err += char_distance(r, h)
        char_total += len(r)
        halluc += len(hw) > 2 * len(rw) + 5
        empty += not h
        for w in rw:
            ref_sc[word_script(w)] += 1
        for w in hw:
            hyp_sc[word_script(w)] += 1
    rt, ht = max(1, sum(ref_sc.values())), max(1, sum(hyp_sc.values()))
    return {
        "wer": 100 * (S + D + I) / N,
        "cer": 100 * char_err / char_total,
        "sub_rate": 100 * S / N, "del_rate": 100 * D / N, "ins_rate": 100 * I / N,
        "ref_latin_pct": 100 * ref_sc["latin"] / rt,
        "hyp_latin_pct": 100 * hyp_sc["latin"] / ht,
        "hyp_devanagari_pct": 100 * hyp_sc["devanagari"] / ht,
        "hyp_other_script_pct": 100 * hyp_sc["other"] / ht,
        "possible_hallucinations": int(halluc), "empty_outputs": int(empty),
    }, per_utt_wer


# Self-test: 1 substitution + 1 deletion + 1 insertion over 4 reference words = 75% WER
_m, _ = compute_metrics(["a b c d"], ["a x c e f"])
assert round(_m["wer"], 2) == 75.0, _m
print("Metric self-test passed:", {k: round(v, 2) for k, v in _m.items() if k.endswith("rate") or k == "wer"})

Metric self-test passed: {'wer': 75.0, 'sub_rate': 50.0, 'del_rate': 0.0, 'ins_rate': 25.0}


## 7. Load the utterances

In [7]:
def read_manifest(path, max_utts=0):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    if max_utts and max_utts < len(rows):
        rng = np.random.default_rng(0)        # fixed subset: the same for every model
        idx = sorted(rng.choice(len(rows), size=max_utts, replace=False))
        rows = [rows[i] for i in idx]
    return rows


ROWS = read_manifest(MANIFEST, MAX_UTTS)
REFS_NORM = [normalize(r["text"]) for r in ROWS]
_keep = [i for i, t in enumerate(REFS_NORM) if t]
ROWS = [ROWS[i] for i in _keep]
REFS_NORM = [REFS_NORM[i] for i in _keep]
HOURS = sum(r["duration"] for r in ROWS) / 3600
print(f"{len(ROWS)} utterances, {HOURS:.2f} h")
for r in ROWS[:3]:
    print(r["utt_id"], r["duration"], "s |", r["text"])

200 utterances, 0.31 h
598753_406yMKxIdSDHRf8H_0008 5.0 s | हम इस background को प्रस्तुति की सभी slides में भी लागू करेंगे
598753_406yMKxIdSDHRf8H_0015 6.0 s | यहाँ दो master pages हैं जिनका इस प्रस्तुति में उपयोग किया गया है
598753_406yMKxIdSDHRf8H_0024 3.0 s | background टैब पर click करें


## 8. Whisper inference

In [8]:
import time
from transformers import WhisperForConditionalGeneration, WhisperProcessor


class WhisperRunner:
    def __init__(self, model_id, device):
        if model_id.endswith(".en"):
            raise ValueError(f"{model_id} is English-only. Use a multilingual checkpoint.")
        self.device = device
        self.dtype = torch.float16 if device == "cuda" else torch.float32
        self.processor = WhisperProcessor.from_pretrained(model_id)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_id)
        self.model = self.model.to(device, dtype=self.dtype).eval()

    def transcribe(self, audios, language, num_beams, max_new_tokens):
        feats = self.processor.feature_extractor(
            audios, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features
        feats = feats.to(self.device, dtype=self.dtype)
        with torch.inference_mode():
            ids = self.model.generate(
                feats,
                language=language,      # "hi" or None (auto-detect)
                task="transcribe",
                num_beams=num_beams,
                do_sample=False,        # temperature 0
                max_new_tokens=max_new_tokens,
            )
        texts = self.processor.batch_decode(ids, skip_special_tokens=True)
        raw = self.processor.batch_decode(ids, skip_special_tokens=False)
        langs = []
        for r in raw:
            m = re.search(r"<\|([a-z]{2,3}(?:-[a-z]+)?)\|>", r)
            langs.append(m.group(1) if m else None)
        return [t.strip() for t in texts], langs


def run_setting(runner, rows, language, num_beams):
    hyps, langs = [], []
    audio_sec = sum(r["duration"] for r in rows)
    t0 = time.perf_counter()
    for i in range(0, len(rows), BATCH_SIZE):
        batch = rows[i:i + BATCH_SIZE]
        texts, detected = runner.transcribe([load_audio(r["audio"]) for r in batch],
                                            language, num_beams, MAX_NEW_TOKENS)
        hyps += texts
        langs += detected
        print(f"\r    {min(i + BATCH_SIZE, len(rows))}/{len(rows)}", end="", flush=True)
    print()
    return hyps, langs, (time.perf_counter() - t0) / max(audio_sec, 1e-9)

## 9. Run the baseline

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "run_info.json").write_text(json.dumps({
    "dataset": "OpenSLR 104 / MUCS 2021 Hindi-English test",
    "n_utterances": len(ROWS), "hours": round(HOURS, 3),
    "settings": {"models": MODELS, "languages": LANGUAGES, "beams": BEAMS,
                 "batch_size": BATCH_SIZE, "max_new_tokens": MAX_NEW_TOKENS,
                 "max_utts": MAX_UTTS, "min_dur": MIN_DUR, "max_dur": MAX_DUR},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__},
    "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
    "normalization": NORMALIZATION_RULES.strip(),
}, indent=2, ensure_ascii=False))

SUMMARY = []
for model_id in MODELS:
    print(f"\n=== {model_id} ===")
    runner = WhisperRunner(model_id, DEVICE)
    for lang in LANGUAGES:
        for beams in BEAMS:
            tag = f"{model_id.split('/')[-1]}__lang-{lang}__beam-{beams}"
            print(f"  {tag}")
            hyps, detected, rtf = run_setting(runner, ROWS, None if lang == "auto" else lang, beams)
            hyps_norm = [normalize(h) for h in hyps]
            m, per_utt = compute_metrics(REFS_NORM, hyps_norm)
            counts = defaultdict(int)
            for d in detected:
                counts[d or "unknown"] += 1
            top = ", ".join(f"{k}:{v}" for k, v in sorted(counts.items(), key=lambda kv: -kv[1])[:4])
            SUMMARY.append({"model": model_id, "language": lang, "beams": beams, **m,
                            "rtf_gpu": rtf, "detected_langs": top, "tag": tag})
            print(f"    WER {m['wer']:.2f} | CER {m['cer']:.2f} | Latin words ref/hyp "
                  f"{m['ref_latin_pct']:.1f}/{m['hyp_latin_pct']:.1f}% | other script "
                  f"{m['hyp_other_script_pct']:.1f}% | RTF {rtf:.3f}")
            with open(OUT_DIR / f"{tag}.jsonl", "w", encoding="utf-8") as f:
                for r, h, rn, hn, d, w in zip(ROWS, hyps, REFS_NORM, hyps_norm, detected, per_utt):
                    f.write(json.dumps({"utt_id": r["utt_id"], "duration": r["duration"],
                                        "ref": r["text"], "hyp": h, "ref_norm": rn, "hyp_norm": hn,
                                        "detected_lang": d, "utt_wer": round(w, 2)},
                                       ensure_ascii=False) + "\n")
    del runner
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print("\nDone.")


=== openai/whisper-tiny ===


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

... (189 outputs left out)

[transformers] Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    192/200

[transformers] Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    200/200
    WER 343.07 | CER 282.11 | Latin words ref/hyp 22.8/0.8% | other script 0.7% | RTF 0.081

Done.


## 10. Results table

- **WER / CER / Sub / Del / Ins** are percentages of the reference words or characters.
- **Latin % (hyp)**: compare with the reference value printed below the table. If it is much lower, Whisper writes English words in Devanagari.
- **Other script %**: words in a third script (for example Urdu). This is a wrong-script error.

In [10]:
import pandas as pd

df = pd.DataFrame(SUMMARY)
cols = ["model", "language", "beams", "wer", "cer", "sub_rate", "del_rate", "ins_rate",
        "hyp_latin_pct", "hyp_devanagari_pct", "hyp_other_script_pct",
        "possible_hallucinations", "empty_outputs", "rtf_gpu", "detected_langs"]
df[cols].to_csv(OUT_DIR / "summary.csv", index=False)
try:
    table_text = df[cols].round(2).to_markdown(index=False)   # needs the "tabulate" package
except ImportError:
    table_text = df[cols].round(2).to_string(index=False)
(OUT_DIR / "summary.md").write_text(table_text, encoding="utf-8")
print(f"Reference: {SUMMARY[0]['ref_latin_pct']:.1f}% of words are in Latin script.")
display(df[cols].round(2))

Reference: 22.8% of words are in Latin script.


,model,language,beams,wer,cer,sub_rate,del_rate,ins_rate,hyp_latin_pct,hyp_devanagari_pct,hyp_other_script_pct,possible_hallucinations,empty_outputs,rtf_gpu,detected_langs
0,openai/whisper-tiny,hi,1,239.71,199.13,65.21,26.43,148.07,50.19,47.41,0.06,26,7,0.03,"unknown:192, hi:8"
1,openai/whisper-tiny,auto,1,234.79,196.72,65.43,26.25,143.12,51.62,45.74,0.27,26,8,0.03,"unknown:192, hi:8"
2,openai/whisper-base,hi,1,246.35,233.01,64.45,26.43,155.47,57.33,27.37,14.84,32,6,0.04,"unknown:199, ur:1"
3,openai/whisper-base,auto,1,224.44,202.11,63.79,27.18,133.47,54.96,27.73,16.80,29,4,0.04,"unknown:199, ur:1"
4,openai/whisper-small,hi,1,365.03,300.14,70.87,10.00,284.15,0.14,99.80,0.00,86,0,0.08,unknown:200
5,openai/whisper-small,auto,1,343.07,282.11,70.39,10.71,261.97,0.76,98.49,0.68,79,0,0.08,unknown:200


## 11. Bad examples

In [11]:
SHOW_TAG = SUMMARY[0]["tag"]
N_SHOW = 10

with open(OUT_DIR / f"{SHOW_TAG}.jsonl", encoding="utf-8") as f:
    utts = [json.loads(line) for line in f]
print("Run:", SHOW_TAG)
for u in sorted(utts, key=lambda u: -u["utt_wer"])[:N_SHOW]:
    print(f"\n{u['utt_id']} | {u['duration']} s | WER {u['utt_wer']} | lang {u['detected_lang']}")
    print("  REF:", u["ref_norm"])
    print("  HYP:", u["hyp_norm"])

Run: whisper-tiny__lang-hi__beam-1

598753_406yMKxIdSDHRf8H_0050 | 5.0 s | WER 5000.0 | lang None
  REF: आप capital i के साथ plus sign देखेंगे
  HYP: you can see the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of the second episode of th

## 12. Save results to Google Drive

In [12]:
if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive") / OUT_DIR.name
    shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
    print("Copied to", dest)
else:
    print("SAVE_TO_DRIVE is False. Results are in", OUT_DIR.resolve(),
          "(Colab deletes them when the session ends).")

SAVE_TO_DRIVE is False. Results are in /content/results/baseline_openslr104 (Colab deletes them when the session ends).


In [13]:
# ===== Fine-tuning settings =====
BASE_MODEL = "openai/whisper-small"          # or "openai/whisper-base"
HINGLISH_DATASET = "agarwalayushi/hinglish"
ENGLISH_DATASET = "fixie-ai/librispeech_asr"  # English mix, so English accuracy does not drop
ENGLISH_CONFIG, ENGLISH_TRAIN_SPLIT, ENGLISH_TEST_SPLIT = "clean", "train.100", "test"

TRAIN_HOURS = 50          # Hinglish hours streamed to disk (about 6 GB)
ENGLISH_FRACTION = 0.10   # English clips = 10% of the Hinglish clips (0 = no English)
ENGLISH_TEST_UTTS = 200   # English clips for the English check
VAL_UTTS = 300            # validation clips from held-out speakers
VAL_SPEAKER_MOD = 30      # about 1 speaker in 30 goes to validation only
SHUFFLE_BUFFER = 2000
SEED = 42

LEARNING_RATE = 1e-5
WARMUP_STEPS = 300
MAX_STEPS = 3000
TRAIN_BATCH = 16          # lower to 8 if you get "CUDA out of memory"
GRAD_ACCUM = 1
EVAL_STEPS = 500
SAVE_STEPS = 500
RUN_INITIAL_EVAL = True   # validation WER before training (step 0)

# SHORT TEST RUN FIRST: set TRAIN_HOURS = 1, MAX_STEPS = 50, EVAL_STEPS = SAVE_STEPS = 25, VAL_UTTS = 50

USE_DRIVE = True          # keep checkpoints on Google Drive, so you can resume after a disconnect
FT_DATA_DIR = Path("data/finetune")
MAX_LABEL_LEN = 440       # Whisper decoder limit is 448 tokens

In [17]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/whisper_hinglish") / BASE_MODEL.split("/")[-1]
else:
    RUN_DIR = Path("runs") / BASE_MODEL.split("/")[-1]
CKPT_DIR, FINAL_DIR = RUN_DIR / "checkpoints", RUN_DIR / "final"
RUN_DIR.mkdir(parents=True, exist_ok=True)

processor = WhisperProcessor.from_pretrained(BASE_MODEL)
tokenizer = processor.tokenizer
SOT_ID = tokenizer.convert_tokens_to_ids("<|startoftranscript|>")


def make_labels(text, lang):
    """<|startoftranscript|><|lang|><|transcribe|><|notimestamps|> text <|endoftext|>"""
    prefix = tokenizer.convert_tokens_to_ids(
        ["<|startoftranscript|>", f"<|{lang}|>", "<|transcribe|>", "<|notimestamps|>"])
    body = tokenizer(text, add_special_tokens=False).input_ids
    return prefix + body + [tokenizer.eos_token_id]


_ids = make_labels("यह bash script है", "hi")
print(len(_ids), "tokens ->", tokenizer.decode(_ids))

Mounted at /content/drive
11 tokens -> <|startoftranscript|><|hi|><|transcribe|><|notimestamps|>यह bash script है<|endoftext|>


In [18]:
# All OpenSLR 104 test transcripts (not only the MAX_UTTS subset).
_kdir = find_kaldi_dir(DATA_ROOT)
TEST_TEXTS = {normalize(t) for t in read_kaldi_map(_kdir / "text").values()}
TEST_TEXTS.discard("")
print(f"{len(TEST_TEXTS)} unique test transcripts will be excluded from training.")

2877 unique test transcripts will be excluded from training.


In [19]:
import io, zlib, random, unicodedata
from datasets import load_dataset, Audio


def clean_text(t):
    t = re.sub(r"^\s*<hi-en>\s*", "", t)          # remove the language tag
    t = unicodedata.normalize("NFC", t)
    return re.sub(r"\s+", " ", t).strip()


def decode_audio(audio_dict):
    data = audio_dict.get("bytes")
    if data is None:
        with open(audio_dict["path"], "rb") as f:
            data = f.read()
    audio, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        from scipy.signal import resample_poly
        g = np.gcd(sr, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // g, sr // g).astype("float32")
    return audio


def is_val_speaker(spk):
    return zlib.crc32(str(spk).encode()) % VAL_SPEAKER_MOD == 0


def stream_hinglish():
    train_m, val_m = FT_DATA_DIR / "train_hinglish.jsonl", FT_DATA_DIR / "val.jsonl"
    if train_m.exists() and val_m.exists():
        print("Using existing files:", train_m, val_m)
        return train_m, val_m
    wav_dir = FT_DATA_DIR / "wav_hinglish"
    wav_dir.mkdir(parents=True, exist_ok=True)

    ds = load_dataset(HINGLISH_DATASET, split="train", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))   # decode ourselves with soundfile
    ds = ds.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)

    stats, train_sec, n_val, t0 = defaultdict(int), 0.0, 0, time.time()
    target = TRAIN_HOURS * 3600
    with open(train_m, "w", encoding="utf-8") as ft, open(val_m, "w", encoding="utf-8") as fv:
        for i, ex in enumerate(ds):
            if train_sec >= target and n_val >= VAL_UTTS:
                break
            text = clean_text(ex["text"])
            norm = normalize(text)
            if not norm:
                stats["empty"] += 1; continue
            if norm in TEST_TEXTS:
                stats["in_openslr_test"] += 1; continue
            spk = str(ex.get("source", ""))
            val = is_val_speaker(spk)
            if (val and n_val >= VAL_UTTS) or (not val and train_sec >= target):
                stats["not_needed"] += 1; continue
            if len(make_labels(text, "hi")) > MAX_LABEL_LEN:
                stats["text_too_long"] += 1; continue
            audio = decode_audio(ex["audio"])
            dur = len(audio) / SAMPLE_RATE
            if not (MIN_DUR <= dur <= MAX_DUR):
                stats["bad_duration"] += 1; continue
            path = wav_dir / f"{i:07d}.wav"
            sf.write(str(path), audio, SAMPLE_RATE, subtype="PCM_16")
            row = {"audio": str(path), "text": text, "lang": "hi",
                   "speaker": spk, "duration": round(dur, 3)}
            (fv if val else ft).write(json.dumps(row, ensure_ascii=False) + "\n")
            if val:
                n_val += 1
            else:
                train_sec += dur
            if i % 1000 == 0:
                print(f"\r  read {i} | train {train_sec/3600:.2f}/{TRAIN_HOURS} h | "
                      f"val {n_val}/{VAL_UTTS} | {time.time()-t0:.0f} s", end="", flush=True)
    print(f"\nDone. Train {train_sec/3600:.2f} h, val {n_val} clips. Skipped: {dict(stats)}")
    return train_m, val_m


TRAIN_HI_MANIFEST, VAL_MANIFEST = stream_hinglish()

README.md:   0%|          | 0.00/6.01k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/816 [00:00<?, ?it/s]

dataset_infos.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

  read 5000 | train 8.90/50 h | val 206/300 | 122 s

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1511 > 1024). Running this sequence through the model will result in indexing errors


  read 23000 | train 49.25/50 h | val 300/300 | 393 s
Done. Train 50.00 h, val 300 clips. Skipped: {'in_openslr_test': 85, 'bad_duration': 1065, 'text_too_long': 237, 'empty': 7, 'not_needed': 109}


In [20]:
def stream_english(split, n_clips, name, shuffle=True):
    out_m = FT_DATA_DIR / f"{name}.jsonl"
    if out_m.exists():
        print("Using existing file:", out_m)
        return out_m
    wav_dir = FT_DATA_DIR / f"wav_{name}"
    wav_dir.mkdir(parents=True, exist_ok=True)
    ds = load_dataset(ENGLISH_DATASET, ENGLISH_CONFIG, split=split, streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
    n, text_col = 0, None
    with open(out_m, "w", encoding="utf-8") as f:
        for i, ex in enumerate(ds):
            if n >= n_clips:
                break
            text_col = text_col or next(c for c in ("text", "sentence", "transcript") if c in ex)
            text = re.sub(r"\s+", " ", ex[text_col]).strip().lower()  # LibriSpeech is UPPERCASE
            audio = decode_audio(ex["audio"])
            dur = len(audio) / SAMPLE_RATE
            if not text or not (MIN_DUR <= dur <= MAX_DUR) or len(make_labels(text, "en")) > MAX_LABEL_LEN:
                continue
            path = wav_dir / f"{i:07d}.wav"
            sf.write(str(path), audio, SAMPLE_RATE, subtype="PCM_16")
            f.write(json.dumps({"audio": str(path), "text": text, "lang": "en",
                                "duration": round(dur, 3)}, ensure_ascii=False) + "\n")
            n += 1
    print(f"{name}: {n} clips")
    return out_m


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


n_hi_train = len(read_jsonl(TRAIN_HI_MANIFEST))
TRAIN_EN_MANIFEST = EN_TEST_MANIFEST = None
try:
    if ENGLISH_FRACTION > 0:
        TRAIN_EN_MANIFEST = stream_english(ENGLISH_TRAIN_SPLIT, int(ENGLISH_FRACTION * n_hi_train), "train_english")
    EN_TEST_MANIFEST = stream_english(ENGLISH_TEST_SPLIT, ENGLISH_TEST_UTTS, "test_english", shuffle=False)
except Exception as err:
    print("WARNING: English data failed, continuing without it:", err)

README.md:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/84 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/84 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

train_english: 2169 clips


Resolving data files:   0%|          | 0/84 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/84 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

test_english: 200 clips


In [21]:
train_rows = read_jsonl(TRAIN_HI_MANIFEST)
if TRAIN_EN_MANIFEST:
    train_rows += read_jsonl(TRAIN_EN_MANIFEST)
random.Random(SEED).shuffle(train_rows)
val_rows = read_jsonl(VAL_MANIFEST)

train_spk = {r.get("speaker") for r in train_rows if r["lang"] == "hi"}
val_spk = {r.get("speaker") for r in val_rows}
assert not (train_spk & val_spk), "Speaker overlap between train and validation!"
print(f"Train: {len(train_rows)} clips, {sum(r['duration'] for r in train_rows)/3600:.2f} h "
      f"({sum(r['lang']=='en' for r in train_rows)} English) | "
      f"Val: {len(val_rows)} clips, {len(val_spk)} speakers")


class ASRDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        feats = processor.feature_extractor(
            load_audio(r["audio"]), sampling_rate=SAMPLE_RATE).input_features[0]
        return {"input_features": feats, "labels": make_labels(r["text"], r["lang"])}


class Collator:
    def __call__(self, features):
        input_features = torch.tensor(np.stack([f["input_features"] for f in features]))
        batch = tokenizer.pad({"input_ids": [f["labels"] for f in features]}, return_tensors="pt")
        labels = batch["input_ids"].masked_fill(batch["attention_mask"].ne(1), -100)
        if (labels[:, 0] == SOT_ID).all():   # the model adds <|startoftranscript|> itself
            labels = labels[:, 1:]
        return {"input_features": input_features, "labels": labels}


train_ds, val_ds, collator = ASRDataset(train_rows), ASRDataset(val_rows), Collator()
_b = collator([train_ds[0], train_ds[1]])
print({k: tuple(v.shape) for k, v in _b.items()})

Train: 23867 clips, 57.60 h (2169 English) | Val: 300 clips, 5 speakers
{'input_features': (2, 80, 3000), 'labels': (2, 158)}


In [22]:
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.config.use_cache = False               # required with gradient checkpointing
model.config.forced_decoder_ids = None
model.generation_config.forced_decoder_ids = None
model.generation_config.language = "hi"      # validation is Hinglish
model.generation_config.task = "transcribe"
model.config.apply_spec_augment = True       # SpecAugment: masks parts of the spectrogram
model.config.mask_time_prob = 0.05
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

242M parameters


In [23]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer


def trainer_metrics(pred):
    pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    pred_ids = np.where(pred_ids == -100, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(pred.label_ids == -100, tokenizer.pad_token_id, pred.label_ids)
    preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pairs = [(normalize(r), normalize(p)) for r, p in zip(refs, preds) if normalize(r)]
    m, _ = compute_metrics([r for r, _ in pairs], [p for _, p in pairs])
    return {"wer": m["wer"], "cer": m["cer"], "ins_rate": m["ins_rate"],
            "hyp_latin_pct": m["hyp_latin_pct"]}


training_args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=TRAIN_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=(DEVICE == "cuda"),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_NEW_TOKENS,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

_kw = dict(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
           data_collator=collator, compute_metrics=trainer_metrics)
try:
    trainer = Seq2SeqTrainer(**_kw, processing_class=processor.feature_extractor)  # new versions
except TypeError:
    trainer = Seq2SeqTrainer(**_kw, tokenizer=processor.feature_extractor)         # old versions

In [24]:
ckpts = sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1])) if CKPT_DIR.exists() else []
last_ckpt = str(ckpts[-1]) if ckpts else None
print("Resume from:", last_ckpt or "start")

if RUN_INITIAL_EVAL and last_ckpt is None:
    print("Validation before training:", trainer.evaluate())

trainer.train(resume_from_checkpoint=last_ckpt)

trainer.save_model(str(FINAL_DIR))          # best checkpoint (lowest validation WER)
processor.save_pretrained(str(FINAL_DIR))
(RUN_DIR / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=2))
print("Saved to", FINAL_DIR)

Resume from: start


Training Loss,Validation Loss,Step,Wer,Cer,Ins Rate,Hyp Latin Pct
No log,1.390634,0,253.600678,196.750329,178.395933,0.507024


Validation before training: {'eval_loss': 1.3906338214874268, 'eval_wer': 253.60067777463993, 'eval_cer': 196.75032880315652, 'eval_ins_rate': 178.3959333521604, 'eval_hyp_latin_pct': 0.5070244005492764}


Step,Training Loss,Validation Loss,Wer,Cer,Ins Rate,Hyp Latin Pct
500,0.485674,0.464435,49.985880,31.039018,9.827732,6.341069
1000,0.385504,0.393150,43.547021,27.767427,8.133296,9.128403
1500,0.359242,0.363217,42.982208,26.572775,10.081898,6.370450
2000,0.322377,0.345628,38.068342,22.068172,6.664784,7.810321
2500,0.288156,0.336776,34.651228,19.503507,4.462016,8.319039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/whisper_hinglish/whisper-small/final


In [25]:
del trainer, model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

FT_NAME = f"{BASE_MODEL.split('/')[-1]}-hinglish-ft"
ft_runner = WhisperRunner(str(FINAL_DIR), DEVICE)

# 1) OpenSLR 104 test (Hinglish), forced Hindi, greedy: same setting as the baseline row
tag = f"{FT_NAME}__lang-hi__beam-1"
hyps, detected, rtf = run_setting(ft_runner, ROWS, "hi", 1)
hyps_norm = [normalize(h) for h in hyps]
m, per_utt = compute_metrics(REFS_NORM, hyps_norm)
SUMMARY.append({"model": FT_NAME, "language": "hi", "beams": 1, **m,
                "rtf_gpu": rtf, "detected_langs": "forced hi", "tag": tag})
with open(OUT_DIR / f"{tag}.jsonl", "w", encoding="utf-8") as f:
    for r, h, rn, hn, w in zip(ROWS, hyps, REFS_NORM, hyps_norm, per_utt):
        f.write(json.dumps({"utt_id": r["utt_id"], "duration": r["duration"], "ref": r["text"],
                            "hyp": h, "ref_norm": rn, "hyp_norm": hn, "utt_wer": round(w, 2)},
                           ensure_ascii=False) + "\n")

# 2) English check: base model vs fine-tuned model on LibriSpeech test-clean clips
english_rows = []
if EN_TEST_MANIFEST:
    en_rows = read_jsonl(EN_TEST_MANIFEST)
    en_refs = [normalize(r["text"]) for r in en_rows]
    for name, runner in [(BASE_MODEL, WhisperRunner(BASE_MODEL, DEVICE)), (FT_NAME, ft_runner)]:
        h, _, _ = run_setting(runner, en_rows, "en", 1)
        em, _ = compute_metrics(en_refs, [normalize(x) for x in h])
        english_rows.append({"model": name, "english_wer": round(em["wer"], 2),
                             "english_cer": round(em["cer"], 2)})

# 3) Comparison tables
base_short = BASE_MODEL.split("/")[-1]
cmp = pd.DataFrame([s for s in SUMMARY if s["language"] == "hi" and s["beams"] == 1
                    and (s["model"] == BASE_MODEL or s["model"] == FT_NAME)])
cmp_cols = ["model", "wer", "cer", "sub_rate", "del_rate", "ins_rate",
            "hyp_latin_pct", "hyp_other_script_pct", "possible_hallucinations"]
print("OpenSLR 104 test (Hinglish):")
display(cmp[cmp_cols].round(2))
cmp[cmp_cols].round(2).to_csv(RUN_DIR / "comparison_openslr104.csv", index=False)
if english_rows:
    print("English check (LibriSpeech test-clean):")
    display(pd.DataFrame(english_rows))
    pd.DataFrame(english_rows).to_csv(RUN_DIR / "comparison_english.csv", index=False)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    16/200

[transformers] Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    32/200

[transformers] Both `max_new_tokens` (=400) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (73 outputs left out)

    200/200
OpenSLR 104 test (Hinglish):


,model,wer,cer,sub_rate,del_rate,ins_rate,hyp_latin_pct,hyp_other_script_pct,possible_hallucinations
0,openai/whisper-small,365.03,300.14,70.87,10.00,284.15,0.14,0.0,86
1,whisper-small-hinglish-ft,83.62,73.00,40.19,10.09,33.33,5.42,0.0,7


English check (LibriSpeech test-clean):


,model,english_wer,english_cer
0,openai/whisper-small,3.26,1.11
1,whisper-small-hinglish-ft,3.61,1.15
